# Feature Engineering Exploration

Explore and validate feature transformations — lag windows, rolling statistics,
and time-based encodings — before hardening them in `transformer.py`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from src.feature_pipeline.transformer import (
    compute_time_features,
    compute_lag_features,
    compute_rolling_features,
    compute_change_rate,
    compute_aqi_category_col,
    transform,
)

print('Transformer modules loaded')

## 1. Load Raw Data

In [ ]:
from src.hopsworks_setup import get_feature_store
from src.training_pipeline.loader import load_feature_view

fs = get_feature_store()
df, _ = load_feature_view(fs)
print(f'Data: {df.shape}')
df.head(3)

## 2. Lag Correlation Analysis

How correlated is AQI at time t with AQI at t-1h, t-3h, t-6h, t-12h, t-24h, t-48h?

In [ ]:
lags_to_test = [1, 3, 6, 12, 24, 48, 72]

for city in df['city'].unique()[:3]:
    city_df = df[df['city'] == city].sort_values('timestamp').copy()
    correlations = {}
    for lag in lags_to_test:
        city_df[f'lag_{lag}'] = city_df['aqi'].shift(lag)
        corr = city_df['aqi'].corr(city_df[f'lag_{lag}'])
        correlations[f'{lag}h'] = corr
    
    corr_series = pd.Series(correlations)
    fig = px.bar(x=corr_series.index, y=corr_series.values,
                 title=f'Autocorrelation — {city}',
                 labels={'x': 'Lag', 'y': 'Pearson r'})
    fig.update_layout(height=300)
    fig.show()

## 3. Optimal Rolling Window Size

Test different rolling windows and compare their predictive power.

In [ ]:
windows = [3, 6, 12, 24, 48]

for w in windows:
    rolled = compute_rolling_features(
        df[['city', 'timestamp', 'aqi']].copy(),
        windows=(w,)
    )
    col = f'aqi_rolling_mean_{w}h'
    corr = rolled['aqi'].corr(rolled[col])
    print(f'Rolling {w}h: correlation with AQI = {corr:.4f}')

## 4. Full Transform Pipeline Test

In [ ]:
engineered = transform(df, pd.DataFrame())  # weather is already in df
print(f'Engineered features: {engineered.shape}')
print(f'New columns: {[c for c in engineered.columns if c not in df.columns]}')
engineered.head()

## 5. Feature Coverage Check

In [ ]:
engineered.describe(include='all').T